# Good Notebook 8: Prometheus on ARC-AGI-3

## Applying Goodian & Hofstadterian Principles to Interactive Reasoning

**Last updated: 2026-03-27**

---

### What is ARC-AGI-3?

ARC-AGI-3 is the **first fully interactive** benchmark in the ARC-AGI series:

| Benchmark | Format | Agent role |
|-----------|--------|------------|
| ARC-AGI-1/2 | Static grid input/output pairs | Passive — infer transformation rule |
| **ARC-AGI-3** | **Interactive turn-based environments** | **Active — explore, discover, adapt** |

ARC-AGI-3 presents **hundreds of hand-crafted game environments** with:
- **No stated goals** — the agent must discover what the winning condition is
- **No stated rules** — the agent must infer environment dynamics from experience
- **Progressive difficulty** — levels escalate as the agent advances
- **Human benchmark: 100% · Frontier AI benchmark: 0.26%**

The five evaluation dimensions:

| Dimension | What it measures |
|-----------|------------------|
| **Exploration** | Does the agent explore effectively? |
| **Percept → Plan → Action** | Can it form and execute plans? |
| **Memory** | Does prior experience improve performance? |
| **Goal Acquisition** | Can it discover hidden objectives? |
| **Alignment** | Does it pursue only intended goals? |

---

### Prometheus Principles Applied

#### I.J. Good (1965) — Recursive Self-Improvement

> *"The first ultraintelligent machine … will design even better machines. There will then unquestionably be an 'intelligence explosion'."*

ARC-AGI-3 demands **online self-improvement**: the agent must improve its own
exploration strategy, world-model, and goal-inference mechanism *during* interaction
— not from pre-training.  Good's **probabilistic synaptic mutation** is applied to
the exploration policy:

- **Upward mutation**: reward-positive strategies gain probability mass
- **Downward mutation**: reward-negative strategies lose probability mass

#### Douglas Hofstadter (1979, 2007) — Strange Loops & Isomorphism

> *"Meaning is an isomorphism between internal representations and external reality."*

Three Hofstadterian principles are simultaneously active in the agent:

1. **Isomorphism** — the world-model strives to be a structure-preserving map
   of the hidden environment dynamics. Isomorphism fidelity is our metric for
   the *Percept→Plan→Action* dimension.

2. **Strange loop** — the agent's world-model shapes its goal-inferrer, which
   shapes its exploration policy, which generates data that updates the
   world-model. Circular causality is the Hofstadterian strange loop.

3. **Tangled hierarchy** — goal-inference (upper level) reads from the action
   history produced by the planner (lower level), but also rewrites that
   planner's objective function.

---

### WP71 Module Map

```
ARC3Observation          ← immutable env snapshot
ARC3Action               ← 7-type canonical action space
ARC3WorldModel           ← Hofstadter isomorphism: env dynamics
ARC3GoalInferrer         ← Hofstadter isomorphism: goal discovery
ARC3ExplorationPolicy    ← Good's probabilistic synaptic mutation
ARC3StrangeLoopAgent     ← all components coupled in a strange loop
ARC3Benchmark            ← multi-game multi-episode evaluator
```

### Sections

1. **Setup** — install & import
2. **The ARC-AGI-3 Action Space** — 7 canonical action types
3. **Hofstadter's Isomorphism: World-Model Learning**
4. **Hofstadter's Goal Discovery: The Inferrer**
5. **Good's Synaptic Mutation: Exploration Policy**
6. **The Strange Loop: All Components Coupled**
7. **Full Benchmark: 4 Game Types × 15 Episodes**
8. **Intelligence Explosion on ARC-AGI-3**
9. **ARC-AGI-3 vs ARC-AGI-1/2: Capability Comparison**
10. **Exit Criteria Verification (WP71)**
11. **Live ARC-AGI-3 Benchmarking via the Official API**


---

## Section 1: Setup


In [ ]:
# ── Colab / local setup ──────────────────────────────────────────────────────
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print('Cloning Prometheus repository...')
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    else:
        os.system('git -C Prometheus_v0_PoC pull origin wp16-notebook-only')
    os.system('pip install -q -e Prometheus_v0_PoC/')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, '..')

import json
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

# ── WP71 imports ─────────────────────────────────────────────────────────────
from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer, ARC3ExplorationPolicy,
    ARC3StrangeLoopAgent, ARC3Benchmark, ARC3BenchmarkReport,
    verify_wp71_exit_criteria, _SyntheticARCGame, _ACTION_TYPES,
)

print('WP71 ARC-AGI-3 module loaded.')
print(f'Canonical action types ({len(_ACTION_TYPES)}): {_ACTION_TYPES}')

---

## Section 2: The ARC-AGI-3 Action Space

ARC-AGI-3 defines **seven canonical action types** across all game environments:

| Action | Parameters | Example use |
|--------|-----------|-------------|
| `move_up/down/left/right` | — | Navigate a grid cell |
| `rotate` | — | Rotate a selected object |
| `place` | x, y | Place an object at coordinates |
| `undo` | — | Reverse the last action |

This standardised action space means the **same agent architecture** can operate
across hundreds of different game environments — a critical requirement for
generalisation.


In [ ]:
def visualize_action_space():
    """Visualise the 7 canonical ARC-AGI-3 action types."""
    fig, axes = plt.subplots(1, 7, figsize=(20, 4))
    fig.suptitle('ARC-AGI-3: The 7 Canonical Action Types', fontsize=16, fontweight='bold')

    action_info = [
        ('move_up',    '↑', '#3498db', 'Move agent\nupward'),
        ('move_down',  '↓', '#3498db', 'Move agent\ndownward'),
        ('move_left',  '←', '#3498db', 'Move agent\nleftward'),
        ('move_right', '→', '#3498db', 'Move agent\nrightward'),
        ('rotate',     '↻', '#e67e22', 'Rotate selected\nobject'),
        ('place',      '✦', '#e74c3c', 'Place object at\n(x, y) coords'),
        ('undo',       '↩', '#2ecc71', 'Reverse last\naction'),
    ]

    for ax, (atype, symbol, colour, desc) in zip(axes, action_info):
        circle = plt.Circle((0.5, 0.65), 0.28, color=colour, alpha=0.85)
        ax.add_patch(circle)
        ax.text(0.5, 0.65, symbol, ha='center', va='center',
                fontsize=28, fontweight='bold', color='white',
                transform=ax.transAxes)
        ax.text(0.5, 0.22, atype, ha='center', va='center',
                fontsize=9, fontweight='bold', transform=ax.transAxes)
        ax.text(0.5, 0.05, desc, ha='center', va='center',
                fontsize=7.5, color='#555', transform=ax.transAxes)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

        # Verify the action constructs without error
        if atype == 'place':
            a = ARC3Action(action_type=atype, x=3, y=2)
        else:
            a = ARC3Action(action_type=atype)
        assert a.action_type == atype

    plt.tight_layout()
    plt.show()
    print('All 7 action types validated.')

visualize_action_space()

---

## Section 3: Hofstadter's Isomorphism — World-Model Learning

> *"Meaning is an isomorphism between internal representations and external reality."*
> — Douglas Hofstadter, *Gödel, Escher, Bach*

The `ARC3WorldModel` builds an internal transition table from experience:

```
T(state, action) → next_state
R(state, action) → expected_reward
```

**Isomorphism fidelity** measures how faithfully this internal model maps to the
true environment dynamics — i.e. what fraction of predicted transitions are correct.

A fidelity of 0 means the model is no better than random;
a fidelity of 1 means the model perfectly predicts the environment.

This directly implements Hofstadter's claim: **intelligence is the process of
building meaning through structure-preserving internal representations**.


In [ ]:
def visualize_world_model_learning():
    """Show isomorphism fidelity growing as the world-model accumulates experience."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Hofstadter's Isomorphism: World-Model Fidelity vs Experience",
                 fontsize=15, fontweight='bold')

    game_types = ['navigate', 'sort', 'count', 'mirror']
    colours    = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
    n_steps    = 80

    for game_type, colour in zip(game_types, colours):
        wm = ARC3WorldModel()
        env = _SyntheticARCGame(game_type, grid_size=5, max_steps=n_steps, seed=0)
        obs = env.reset()
        fidelities = [0.0]
        coverages  = [0.0]

        for step in range(n_steps):
            action = ARC3Action(
                action_type='place' if step % 5 == 0 else random.choice(_ACTION_TYPES[:5]),
                x=random.randint(0, 4), y=random.randint(0, 4)
            )
            next_obs, reward = env.step(action)
            wm.update(obs, action, next_obs, reward)
            fidelities.append(wm.isomorphism_fidelity)
            coverages.append(min(wm.coverage, 1.0))
            obs = next_obs
            if obs.done:
                break

        ax1.plot(fidelities, label=game_type, color=colour, linewidth=2.5)
        ax2.plot(coverages,  label=game_type, color=colour, linewidth=2.5)

    ax1.set_xlabel('Interaction steps', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Isomorphism fidelity', fontsize=12, fontweight='bold')
    ax1.set_title('Isomorphism Fidelity\n(fraction of transitions correctly predicted)',
                  fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1.05)
    ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='50% baseline')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel('Interaction steps', fontsize=12, fontweight='bold')
    ax2.set_ylabel('State–action coverage', fontsize=12, fontweight='bold')
    ax2.set_title('Transition Table Coverage\n(exploration breadth)',
                  fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 1.05)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\nHofstadter Isomorphism Summary:')
    print('  Fidelity starts at 0 (no knowledge) and rises as the')
    print('  world-model accumulates verified transitions.')
    print('  A high fidelity score = high-meaning internal representation.')

visualize_world_model_learning()

---

## Section 4: Hofstadter's Goal Discovery — The Goal Inferrer

ARC-AGI-3 gives **no stated goals** — the agent must discover what it is trying
to achieve.  This is the *Goal Acquisition* evaluation dimension.

The `ARC3GoalInferrer` maintains a probability distribution over six goal hypotheses:

| Hypothesis | Description |
|------------|-------------|
| `maximise_score` | Accumulate the highest numeric score |
| `reach_target` | Move a specific colour to a specific cell |
| `fill_pattern` | Make the grid match a target pattern |
| `clear_colour` | Eliminate all instances of a particular colour |
| `survive` | Remain alive (avoid premature termination) |
| `exploration` | Visit as many unique states as possible |

**Isomorphism fidelity** for goal inference is measured as
`1 - H(p) / H_max` — how peaked is the distribution?
A peaked distribution = the agent has converged on a single goal hypothesis.


In [ ]:
def visualize_goal_inference():
    """Show goal-hypothesis probabilities evolving across different game types."""

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle("Hofstadter's Goal Isomorphism: Inferring Hidden Objectives",
                 fontsize=15, fontweight='bold')

    game_types = ['navigate', 'sort', 'count', 'mirror']
    hypothesis_colours = {
        'maximise_score': '#e74c3c',
        'reach_target':   '#3498db',
        'fill_pattern':   '#2ecc71',
        'clear_colour':   '#9b59b6',
        'survive':        '#e67e22',
        'exploration':    '#1abc9c',
    }

    for ax, game_type in zip(axes.flat, game_types):
        gi = ARC3GoalInferrer()
        env = _SyntheticARCGame(game_type, grid_size=5, max_steps=60, seed=7)
        obs = env.reset()
        prev_obs = None

        history = {h: [] for h in gi._HYPOTHESES}
        fidelity_hist = []

        for step in range(50):
            action = ARC3Action(
                action_type='place' if game_type in ('sort', 'count') else
                            random.choice(_ACTION_TYPES[:4]),
                x=random.randint(0, 4), y=random.randint(0, 4)
            )
            next_obs, reward = env.step(action)
            gi.observe(next_obs, prev_obs, reward)

            for h in gi._HYPOTHESES:
                history[h].append(gi._confidence[h])
            fidelity_hist.append(gi.isomorphism_fidelity)

            prev_obs = obs
            obs = next_obs
            if obs.done:
                break

        steps = range(len(fidelity_hist))
        for h, probs in history.items():
            ax.plot(steps, probs[:len(fidelity_hist)], label=h,
                    color=hypothesis_colours[h], linewidth=2,
                    linestyle='-' if h in ('maximise_score', 'exploration') else '--')

        ax2 = ax.twinx()
        ax2.plot(steps, fidelity_hist, color='black', linewidth=1.5,
                 linestyle=':', alpha=0.5, label='Isomorphism fidelity')
        ax2.set_ylabel('Fidelity', fontsize=9)
        ax2.set_ylim(0, 1)

        ax.set_title(f'Game: {game_type.upper()}\n'
                     f'Inferred: {gi.most_likely_goal} '
                     f'(conf={gi.goal_confidence:.2f})',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Step', fontsize=10)
        ax.set_ylabel('Hypothesis probability', fontsize=10)
        ax.set_ylim(0, 1)
        ax.legend(loc='upper left', fontsize=7.5, ncol=2)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\nGoal Acquisition Summary:')
    print('  Different game types drive different goal hypotheses to prominence.')
    print('  The distribution peaks (high fidelity) when the agent has')
    print('  converged on the most likely hidden objective.')

visualize_goal_inference()

---

## Section 5: Good's Synaptic Mutation — Exploration Policy

> *"There are two kinds of probabilistic synaptic mutation: upward mutation
> and downward mutation."*
> — I.J. Good, *Speculations Concerning the First Ultraintelligent Machine* (1965)

The `ARC3ExplorationPolicy` maintains a probability distribution over five
exploration strategies.  After each episode:

- **Upward mutation**: strategies with above-average reward get a probability boost (Good's α)
- **Downward mutation**: strategies with below-average reward get a probability reduction

This is the **exploration equivalent** of the weight-update in Good Notebook 2
(dynamic ARC-AGI-1/2 solver) — but now applied to the *meta-level* problem of
*how to explore* an unknown environment.

| Strategy | Description |
|----------|-------------|
| `random_walk` | Uniformly random actions |
| `model_guided` | Use world-model's expected-reward estimates |
| `goal_directed` | Bias actions toward the inferred goal |
| `analogy_based` | Mirror actions from similar solved episodes |
| `epsilon_greedy` | Greedy + ε random exploration |


In [ ]:
def visualize_good_synaptic_mutation():
    """Show Good's upward/downward mutation evolving strategy probabilities."""

    n_episodes = 30

    # Simulate: model_guided is objectively best for 'navigate'
    strategy_base_rewards = {
        'random_walk':    0.20,
        'model_guided':   0.75,   # best strategy
        'goal_directed':  0.50,
        'analogy_based':  0.35,
        'epsilon_greedy': 0.45,
    }

    policy = ARC3ExplorationPolicy(mutation_rate=0.08)
    history = [dict(policy._probs)]
    rng = random.Random(42)

    for _ in range(n_episodes):
        for strategy, base in strategy_base_rewards.items():
            # Add noise to simulate real episode variance
            reward = max(0.0, base + rng.gauss(0, 0.1))
            policy.record_episode(strategy, reward)
        policy.mutate()
        history.append(dict(policy._probs))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Good's Probabilistic Synaptic Mutation: Exploration Policy Evolution",
                 fontsize=15, fontweight='bold')

    strategy_colours = {
        'random_walk':    '#e74c3c',
        'model_guided':   '#27ae60',   # winner — shown bold
        'goal_directed':  '#3498db',
        'analogy_based':  '#9b59b6',
        'epsilon_greedy': '#e67e22',
    }

    episodes = range(len(history))
    for strategy, colour in strategy_colours.items():
        probs = [h[strategy] for h in history]
        lw = 4 if strategy == 'model_guided' else 2
        alpha = 1.0 if strategy == 'model_guided' else 0.6
        ax1.plot(episodes, probs, label=strategy, color=colour,
                 linewidth=lw, alpha=alpha, marker='o', markersize=4)

    ax1.axhline(1.0 / 5, color='gray', linestyle='--', alpha=0.4, label='Uniform prior (0.20)')
    ax1.set_xlabel('Episode', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Strategy probability', fontsize=12, fontweight='bold')
    ax1.set_title('Strategy Probabilities\n(Good\'s Upward/Downward Mutation)',
                  fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1.0)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Annotate upward/downward arrows
    ax1.annotate('Upward mutation\n(model_guided wins)',
                 xy=(n_episodes, history[-1]['model_guided']),
                 xytext=(n_episodes * 0.6, 0.7),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 fontsize=10, color='green', fontweight='bold')
    ax1.annotate('Downward mutation\n(random_walk loses)',
                 xy=(n_episodes, history[-1]['random_walk']),
                 xytext=(n_episodes * 0.6, 0.12),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=10, color='red', fontweight='bold')

    # Bar chart of final distribution
    final = history[-1]
    bars = ax2.bar(list(final.keys()), list(final.values()),
                   color=[strategy_colours[s] for s in final],
                   edgecolor='black', linewidth=1.2, alpha=0.85)
    ax2.axhline(1.0 / 5, color='gray', linestyle='--', alpha=0.5, label='Uniform prior')
    ax2.set_ylabel('Final probability', fontsize=12, fontweight='bold')
    ax2.set_title(f'Final Strategy Distribution\n(after {n_episodes} episodes)',
                  fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 1.0)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_xticklabels(list(final.keys()), rotation=20, ha='right', fontsize=10)

    for bar, (s, p) in zip(bars, final.items()):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{p:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'\nGood\'s Mutation Summary:')
    print(f'  Winning strategy: {policy.winning_strategy}')
    print(f'  Initial prob (uniform): {1/5:.3f}')
    print(f'  Final prob:             {policy._probs[policy.winning_strategy]:.3f}')
    improvement = policy._probs[policy.winning_strategy] - 1/5
    print(f'  Upward mutation delta:  +{improvement:.3f}')
    print('\n  This is Good\'s intelligence explosion at the exploration level:')
    print('  the agent learns HOW to explore, not just what to do.')

visualize_good_synaptic_mutation()

---

## Section 6: The Strange Loop — All Components Coupled

The key insight of Hofstadter's *Gödel, Escher, Bach* is that intelligence
emerges from **strange loops** — systems that cross their own level boundaries
in circular causality.

In the `ARC3StrangeLoopAgent`, the loop runs as follows:

```
Step 1: Observation  →  World-Model update      (lower → middle)
Step 2: World-Model  →  Goal-Inferrer update    (middle → upper)
Step 3: Goal-Inferrer → Policy mutation signal  (upper → middle)
Step 4: Policy       →  Action selection        (middle → lower)
Step 5: Action       →  Environment step        (lower level)
         ↑__________________________|  (observation feedback closes the loop)
```

**Entanglement index**: measures how tightly the world-model and policy are
coupled.  Formula: `2·min(wm_coverage, policy_diversity) / (wm_coverage + policy_diversity)`

- Index = 0: one component dominates (nested, not tangled)
- Index = 1: perfectly balanced (Hofstadterian tangled hierarchy)


In [ ]:
def visualize_strange_loop():
    """Visualise the strange-loop architecture and entanglement index over episodes."""

    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

    ax_loop  = fig.add_subplot(gs[0, 0])   # Loop diagram
    ax_ent   = fig.add_subplot(gs[0, 1])   # Entanglement over episodes
    ax_iso   = fig.add_subplot(gs[1, 0])   # Isomorphism fidelity over episodes
    ax_goal  = fig.add_subplot(gs[1, 1])   # Goal confidence over episodes

    # ── Loop diagram ──────────────────────────────────────────────────────────
    components = [
        (0.50, 0.88, 'Environment\n(ARC-AGI-3 Game)', '#2c3e50', 'white'),
        (0.15, 0.50, 'World\nModel', '#2980b9', 'white'),
        (0.50, 0.12, 'Exploration\nPolicy', '#27ae60', 'white'),
        (0.85, 0.50, 'Goal\nInferrer', '#8e44ad', 'white'),
    ]
    arrows = [
        (0.50, 0.82, 0.20, 0.58, 'observation', '#2980b9'),
        (0.18, 0.42, 0.45, 0.18, 'dynamics→\ngoal signal', '#8e44ad'),
        (0.55, 0.12, 0.80, 0.42, 'goal→policy\nbias', '#27ae60'),
        (0.80, 0.58, 0.55, 0.82, 'action', '#2c3e50'),
    ]

    for x, y, label, bg, fg in components:
        rect = mpatches.FancyBboxPatch(
            (x - 0.12, y - 0.09), 0.24, 0.18,
            boxstyle='round,pad=0.02', facecolor=bg, edgecolor='black',
            linewidth=2, transform=ax_loop.transAxes, zorder=3
        )
        ax_loop.add_patch(rect)
        ax_loop.text(x, y, label, ha='center', va='center',
                     fontsize=10, fontweight='bold', color=fg,
                     transform=ax_loop.transAxes, zorder=4)

    for x1, y1, x2, y2, label, colour in arrows:
        ax_loop.annotate('', xy=(x2, y2), xytext=(x1, y1),
                         xycoords='axes fraction', textcoords='axes fraction',
                         arrowprops=dict(arrowstyle='->', color=colour, lw=2.5))
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax_loop.text(mx, my, label, ha='center', va='center',
                     fontsize=8, color=colour, fontweight='bold',
                     transform=ax_loop.transAxes)

    ax_loop.text(0.5, 0.50, 'STRANGE\nLOOP', ha='center', va='center',
                 fontsize=16, fontweight='bold', color='#c0392b', alpha=0.4,
                 transform=ax_loop.transAxes)
    ax_loop.set_title("Hofstadter's Strange Loop Architecture",
                      fontsize=12, fontweight='bold')
    ax_loop.set_xlim(0, 1); ax_loop.set_ylim(0, 1); ax_loop.axis('off')

    # ── Run episodes and collect metrics ──────────────────────────────────────
    game_types = ['navigate', 'sort', 'count', 'mirror']
    game_colours = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
    n_episodes = 20

    for game_type, colour in zip(game_types, game_colours):
        agent = ARC3StrangeLoopAgent(
            max_steps_per_episode=30, mutation_rate=0.06, fitness_threshold=0.7
        )
        ent_hist, iso_hist, goal_hist = [], [], []

        for ep in range(n_episodes):
            env = _SyntheticARCGame(game_type, grid_size=5, max_steps=30, seed=ep)
            agent.run_episode(env)
            ent_hist.append(agent.entanglement_index)
            iso_hist.append(agent.world_model.isomorphism_fidelity)
            goal_hist.append(agent.goal_inferrer.goal_confidence)

        ax_ent.plot(range(1, n_episodes + 1), ent_hist,
                    label=game_type, color=colour, linewidth=2)
        ax_iso.plot(range(1, n_episodes + 1), iso_hist,
                    label=game_type, color=colour, linewidth=2)
        ax_goal.plot(range(1, n_episodes + 1), goal_hist,
                     label=game_type, color=colour, linewidth=2)

    for ax, title, ylabel in [
        (ax_ent,  'Entanglement Index\n(World-Model ↔ Policy coupling)',
                  'Entanglement'),
        (ax_iso,  'World-Model Isomorphism Fidelity\n(Percept→Plan→Action)',
                  'Fidelity'),
        (ax_goal, 'Goal-Inferrer Confidence\n(Goal Acquisition)',
                  'Confidence'),
    ]:
        ax.set_xlabel('Episode', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)

    fig.suptitle("ARC-AGI-3: Hofstadter Strange Loop — Metrics Over Episodes",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    print('\nStrange Loop Summary:')
    print('  Entanglement index rises as world-model and policy co-evolve.')
    print('  Isomorphism fidelity captures how well the agent models env dynamics.')
    print('  Goal confidence tracks convergence on the hidden objective.')

visualize_strange_loop()

---

## Section 7: Full Benchmark — 4 Game Types × 15 Episodes

We now run the complete `ARC3Benchmark` across all four synthetic game types,
evaluating all five ARC-AGI-3 dimensions.


In [ ]:
# Configuration toggle
QUICK_DEMO_MODE = True  # Set False for longer, more stable run

EPISODES_PER_GAME = 15 if QUICK_DEMO_MODE else 40
MAX_STEPS         = 35 if QUICK_DEMO_MODE else 50

print(f'Mode: {"QUICK DEMO" if QUICK_DEMO_MODE else "FULL VALIDATION"}')
print(f'Episodes per game: {EPISODES_PER_GAME}  |  Max steps: {MAX_STEPS}')
print()

bench = ARC3Benchmark(
    game_types=['navigate', 'sort', 'count', 'mirror'],
    episodes_per_game=EPISODES_PER_GAME,
    grid_size=5,
    max_steps=MAX_STEPS,
    fitness_threshold=0.7,
    mutation_rate=0.06,
    seed=42,
)

t0 = time.time()
report = bench.run()
elapsed = time.time() - t0

print(report.summary())
print(f'\nCompleted in {elapsed:.2f}s')

In [ ]:
def visualize_benchmark_report(report: ARC3BenchmarkReport):
    """Visualise the full benchmark report across all five ARC-AGI-3 dimensions."""

    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

    ax_solve  = fig.add_subplot(gs[0, 0])
    ax_score  = fig.add_subplot(gs[0, 1])
    ax_radar  = fig.add_subplot(gs[0, 2], projection='polar')
    ax_iso    = fig.add_subplot(gs[1, 0])
    ax_goal   = fig.add_subplot(gs[1, 1])
    ax_ent    = fig.add_subplot(gs[1, 2])

    games   = [r.game_id for r in report.game_results]
    colours = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']

    # ── Solve rate ────────────────────────────────────────────────────────────
    solve_rates = [r.solve_rate * 100 for r in report.game_results]
    bars = ax_solve.bar(games, solve_rates, color=colours, edgecolor='black',
                        linewidth=1.2, alpha=0.85)
    ax_solve.axhline(report.overall_solve_rate * 100, color='black',
                     linestyle='--', linewidth=2, label='Overall avg')
    ax_solve.set_ylabel('Solve rate (%)', fontsize=11, fontweight='bold')
    ax_solve.set_title('Solve Rate by Game', fontsize=12, fontweight='bold')
    ax_solve.set_ylim(0, 105)
    ax_solve.legend(); ax_solve.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, solve_rates):
        ax_solve.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                      f'{v:.0f}%', ha='center', fontsize=11, fontweight='bold')

    # ── Mean score ────────────────────────────────────────────────────────────
    mean_scores = [r.mean_score for r in report.game_results]
    ax_score.bar(games, mean_scores, color=colours, edgecolor='black',
                 linewidth=1.2, alpha=0.85)
    ax_score.set_ylabel('Mean episode score', fontsize=11, fontweight='bold')
    ax_score.set_title('Mean Score by Game', fontsize=12, fontweight='bold')
    ax_score.grid(True, alpha=0.3, axis='y')

    # ── Radar: 5 ARC-AGI-3 dimensions ────────────────────────────────────────
    dims   = ['Exploration', 'Percept\n→Plan→Action', 'Memory',
               'Goal\nAcquisition', 'Alignment']
    values = [
        report.exploration_score,
        report.mean_isomorphism_fidelity,
        report.memory_score,
        report.mean_goal_confidence,
        report.alignment_score,
    ]
    n = len(dims)
    angles = [i * 2 * math.pi / n for i in range(n)] + [0]
    values_plot = values + [values[0]]

    ax_radar.plot(angles, values_plot, 'o-', linewidth=2.5, color='#2980b9')
    ax_radar.fill(angles, values_plot, alpha=0.25, color='#2980b9')
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(dims, fontsize=9, fontweight='bold')
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title('5 ARC-AGI-3 Dimensions\n(Prometheus WP71)',
                        fontsize=12, fontweight='bold', pad=20)
    ax_radar.grid(True)

    # ── Isomorphism fidelity ──────────────────────────────────────────────────
    iso = [r.final_isomorphism_fidelity for r in report.game_results]
    ax_iso.bar(games, iso, color=colours, edgecolor='black',
               linewidth=1.2, alpha=0.85)
    ax_iso.set_ylabel('Isomorphism fidelity', fontsize=11, fontweight='bold')
    ax_iso.set_title('World-Model Isomorphism\n[Percept→Plan→Action]',
                     fontsize=12, fontweight='bold')
    ax_iso.set_ylim(0, 1.05); ax_iso.grid(True, alpha=0.3, axis='y')

    # ── Goal confidence ───────────────────────────────────────────────────────
    goal = [r.final_goal_confidence for r in report.game_results]
    ax_goal.bar(games, goal, color=colours, edgecolor='black',
                linewidth=1.2, alpha=0.85)
    ax_goal.set_ylabel('Goal confidence', fontsize=11, fontweight='bold')
    ax_goal.set_title('Goal Inferrer Confidence\n[Goal Acquisition]',
                      fontsize=12, fontweight='bold')
    ax_goal.set_ylim(0, 1.05); ax_goal.grid(True, alpha=0.3, axis='y')

    # ── Entanglement index ────────────────────────────────────────────────────
    ent = [r.final_entanglement_index for r in report.game_results]
    ax_ent.bar(games, ent, color=colours, edgecolor='black',
               linewidth=1.2, alpha=0.85)
    ax_ent.set_ylabel('Entanglement index', fontsize=11, fontweight='bold')
    ax_ent.set_title('Strange-Loop Entanglement\n[World-Model ↔ Policy]',
                     fontsize=12, fontweight='bold')
    ax_ent.set_ylim(0, 1.05); ax_ent.grid(True, alpha=0.3, axis='y')

    fig.suptitle('WP71 ARC-AGI-3 Benchmark: Prometheus Strange-Loop Agent',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

visualize_benchmark_report(report)

---

## Section 8: Intelligence Explosion on ARC-AGI-3

Good's (1965) hypothesis: a machine that improves its own performance will
exhibit an *intelligence explosion* — exponential growth in capability.

On ARC-AGI-1/2 we measured this via transformation-fit accuracy (WP44).

On **ARC-AGI-3** we measure it through **goal-acquisition speed** — how many
steps does the agent need before it discovers the hidden goal?

Early episodes: the agent explores randomly, slow to acquire goals.
Later episodes: the world-model, goal-inferrer, and policy have converged —
goal acquisition is dramatically faster.


In [ ]:
def visualize_intelligence_explosion():
    """Compare goal-acquisition speed: early vs late episodes."""

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Good's Intelligence Explosion on ARC-AGI-3\n"
                 "(Goal-Acquisition Speed Over Episodes)",
                 fontsize=15, fontweight='bold')

    game_types   = ['navigate', 'sort', 'count', 'mirror']
    game_colours = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
    n_episodes   = 25

    for game_type, colour in zip(game_types, game_colours):
        agent = ARC3StrangeLoopAgent(
            max_steps_per_episode=40, mutation_rate=0.08, fitness_threshold=0.6
        )
        scores, confidences = [], []

        for ep in range(n_episodes):
            env = _SyntheticARCGame(game_type, grid_size=5, max_steps=40, seed=ep * 7)
            episode = agent.run_episode(env)
            scores.append(episode.total_score)
            confidences.append(agent.goal_inferrer.goal_confidence)

        # Rolling average (window=3)
        def rolling(data, w=3):
            return [sum(data[max(0, i-w+1):i+1]) / len(data[max(0, i-w+1):i+1])
                    for i in range(len(data))]

        axes[0].plot(range(1, n_episodes + 1), rolling(scores),
                     label=game_type, color=colour, linewidth=2.5)
        axes[1].plot(range(1, n_episodes + 1), rolling(confidences),
                     label=game_type, color=colour, linewidth=2.5)

    for ax, title, ylabel in [
        (axes[0], 'Episode Score (3-ep rolling avg)\n'
                  'Rising = Good\'s Intelligence Explosion', 'Episode score'),
        (axes[1], 'Goal Confidence (3-ep rolling avg)\n'
                  'Rising = Faster Goal Acquisition', 'Goal confidence'),
    ]:
        ax.set_xlabel('Episode', fontsize=12, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\nIntelligence Explosion Summary:')
    print('  The agent progressively learns:')
    print('  1. How the environment works (world-model)')
    print('  2. What the hidden goal is (goal-inferrer)')
    print('  3. Which strategy to use (exploration policy)')
    print('  Each episode the loop tightens — this is Good\'s explosion.')

visualize_intelligence_explosion()

---

## Section 9: ARC-AGI-3 vs ARC-AGI-1/2 — Capability Comparison

The Prometheus stack was originally designed for ARC-AGI-1/2 (static grid
transformation).  ARC-AGI-3 requires fundamentally new capabilities.

This section shows **which Prometheus modules were reused, extended, or newly
created** for ARC-AGI-3.


In [ ]:
def visualize_capability_comparison():
    """Side-by-side comparison of Prometheus on ARC-AGI-1/2 vs ARC-AGI-3."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
    fig.suptitle('Prometheus Capability Map: ARC-AGI-1/2 vs ARC-AGI-3',
                 fontsize=15, fontweight='bold')

    # ── ARC-AGI-1/2 stack ─────────────────────────────────────────────────────
    stack_12 = [
        ('ARCGrid / ARCTask', 'Data wrappers', '#3498db'),
        ('ARCTransform (23)', 'Grid operations library', '#3498db'),
        ('ARCProgramSynthesiser', 'Beam-search over transforms', '#2980b9'),
        ('ARCSolver', 'Feature extraction + synthesis', '#2980b9'),
        ('WP38 Analogy Engine', 'Template transfer from solved tasks', '#1abc9c'),
        ('WP34 Proof Tree', 'Best-first search (synthesis prior)', '#1abc9c'),
        ('WP44 Explosion Tracker', 'Intelligence explosion (accuracy)', '#e67e22'),
        ('WP62 ARCBenchmark', 'Multi-task evaluator', '#e74c3c'),
    ]

    # ── ARC-AGI-3 stack ───────────────────────────────────────────────────────
    stack_3 = [
        ('ARC3Action (7 types)', 'Interactive action space [NEW]', '#27ae60'),
        ('ARC3Observation', 'Turn-by-turn env snapshot [NEW]', '#27ae60'),
        ('ARC3WorldModel', 'Isomorphism: dynamics model [NEW]', '#27ae60'),
        ('ARC3GoalInferrer', 'Isomorphism: goal discovery [NEW]', '#27ae60'),
        ('ARC3ExplorationPolicy', 'Good\'s synaptic mutation [NEW]', '#27ae60'),
        ('WP38 Analogy Engine', 'Episode-level analogy transfer [REUSED]', '#1abc9c'),
        ('WP41 Strange Loop', 'Entanglement metric [EXTENDED]', '#f39c12'),
        ('WP71 ARC3Benchmark', 'Interactive multi-game evaluator [NEW]', '#e74c3c'),
    ]

    for ax, stack, title in [
        (ax1, stack_12, 'ARC-AGI-1/2 (WP62)\nStatic Grid Transformation'),
        (ax2, stack_3,  'ARC-AGI-3 (WP71)\nInteractive Agent'),
    ]:
        n = len(stack)
        for i, (name, desc, colour) in enumerate(stack):
            y = 1.0 - (i + 0.5) / n
            rect = mpatches.FancyBboxPatch(
                (0.05, y - 0.06), 0.90, 0.10,
                boxstyle='round,pad=0.01', facecolor=colour, edgecolor='black',
                linewidth=1.5, alpha=0.85, transform=ax.transAxes
            )
            ax.add_patch(rect)
            ax.text(0.50, y, f'{name}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white',
                    transform=ax.transAxes)
            ax.text(0.50, y - 0.035, desc, ha='center', va='center',
                    fontsize=7.5, color='white', alpha=0.9,
                    transform=ax.transAxes)
        ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

    # Legend
    legend_patches = [
        mpatches.Patch(color='#27ae60', label='New (WP71)'),
        mpatches.Patch(color='#f39c12', label='Extended (WP41+71)'),
        mpatches.Patch(color='#1abc9c', label='Reused (WP38)'),
        mpatches.Patch(color='#3498db', label='Foundation (WP62)'),
        mpatches.Patch(color='#2980b9', label='Synthesis (WP34/62)'),
        mpatches.Patch(color='#e74c3c', label='Evaluator'),
    ]
    fig.legend(handles=legend_patches, loc='lower center',
               ncol=3, fontsize=10, frameon=True)

    plt.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()

    print('\nCapability Comparison Summary:')
    print('  ARC-AGI-1/2: 23 static transforms + beam search + analogy')
    print('  ARC-AGI-3:   7 action types + world-model + goal inference')
    print('               + Good\'s mutation + strange loop coupling')
    print()
    print('  The move from passive prediction to active interactive exploration')
    print('  is the key architectural shift — and the reason frontier AIs')
    print('  score 0.26% while humans score 100%.')

visualize_capability_comparison()

---

## Section 10: Exit Criteria Verification (WP71)

All seven measurable exit criteria for WP71 are verified here.


In [ ]:
def run_and_visualize_exit_criteria():
    """Run all seven WP71 exit criteria and produce a pass/fail chart."""

    print('Running WP71 exit criteria...')
    results = verify_wp71_exit_criteria(report)

    criteria_labels = [
        'C1: All 7 action types\nconstruct correctly',
        'C2: World-model learns\ndynamics (fidelity > 0)',
        'C3: Goal-inferrer\nconverges (fidelity ≥ 0.2)',
        'C4: Good\'s upward mutation\n(winning strategy rises)',
        'C5: Agent solves ≥1\n\'navigate\' episode',
        'C6: Entanglement index\n≥ 0.2 after 5 episodes',
        'C7: Report is\nJSON-serialisable',
    ]

    passed = [results[k] for k in sorted(results)]
    colours = ['#27ae60' if p else '#e74c3c' for p in passed]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('WP71 ARC-AGI-3: Exit Criteria Verification',
                 fontsize=15, fontweight='bold')

    bars = ax1.barh(range(len(passed)), [1] * len(passed),
                    color=colours, edgecolor='black', linewidth=1.2, alpha=0.85)
    ax1.set_yticks(range(len(passed)))
    ax1.set_yticklabels(criteria_labels, fontsize=9)
    ax1.set_xticks([])
    ax1.set_title('Exit Criteria', fontsize=12, fontweight='bold')

    for i, (bar, p) in enumerate(zip(bars, passed)):
        ax1.text(0.5, bar.get_y() + bar.get_height() / 2,
                 'PASS' if p else 'FAIL',
                 ha='center', va='center',
                 fontsize=13, fontweight='bold', color='white')

    # Summary pie
    n_pass = sum(passed)
    n_fail = len(passed) - n_pass
    ax2.pie([n_pass, n_fail],
            labels=[f'PASS ({n_pass})', f'FAIL ({n_fail})'],
            colors=['#27ae60', '#e74c3c'],
            autopct='%1.0f%%', startangle=90,
            textprops={'fontsize': 14, 'fontweight': 'bold'})
    ax2.set_title(f'{n_pass}/{len(passed)} Criteria Passed',
                  fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print('\nDetailed results:')
    for key, val in sorted(results.items()):
        status = 'PASS' if val else 'FAIL'
        print(f'  [{status}]  {key}')
    print(f'\n{n_pass}/{len(passed)} criteria passed.')

run_and_visualize_exit_criteria()

---

## Conclusion

### What We Built (WP71)

| Component | Principle | ARC-AGI-3 Dimension |
|-----------|-----------|---------------------|
| `ARC3WorldModel` | Hofstadter isomorphism | Percept → Plan → Action |
| `ARC3GoalInferrer` | Hofstadter isomorphism | Goal Acquisition |
| `ARC3ExplorationPolicy` | Good's synaptic mutation | Exploration |
| Analogy reuse (WP38) | Hofstadter isomorphism | Memory |
| Gödel safety (WP57) | Good's safety governor | Alignment |
| `ARC3StrangeLoopAgent` | Hofstadter strange loop | All five dimensions |

### Why ARC-AGI-3 is Hard

The human–AI gap (100% vs 0.26%) exists because ARC-AGI-3 requires:

1. **Genuine exploration** — not pattern matching from training data
2. **Goal discovery** — inferring what success looks like, without being told
3. **Dynamic self-improvement** — Good's loop, not static inference
4. **Strange-loop self-reference** — the agent must model its own learning process

### Next Steps

- **Connect to the live ARC-AGI-3 API** via the `arc-agi-toolkit` package
- **Scale the world-model** using WP36's transformer policy
- **Apply WP57 Gödel Machine** to verify self-modifications before applying them
- **Run the WP44 explosion tracker** on ARC-AGI-3 level progression data

---

**References**

1. Good, I.J. (1965). *Speculations Concerning the First Ultraintelligent Machine.*
2. Hofstadter, D.R. (1979). *Gödel, Escher, Bach: An Eternal Golden Braid.*
3. Hofstadter, D.R. (2007). *I Am a Strange Loop.*
4. Chollet, F. (2019). *On the Measure of Intelligence.*
5. ARC Prize (2025). *ARC-AGI-3: An Interactive Reasoning Benchmark.* https://arcprize.org


---

## Section 11: Live ARC-AGI-3 Benchmarking via the Official API

The cells below connect Prometheus's `ARC3StrangeLoopAgent` to the **real**
ARC-AGI-3 API using the `arc-agi` toolkit.

### Prerequisites

| Step | Action |
|------|--------|
| 1 | `pip install arc-agi` (handled automatically below) |
| 2 | Register at **arcprize.org/platform** (Google or GitHub login) |
| 3 | Create an API key in your profile → **API Keys** |
| 4 | Paste it in the `ARC_API_KEY` variable below |

> **No API key?**  Three games (`ls20`, `ft09`, `vc33`) are available
> anonymously.  Leave `ARC_API_KEY = ""` to use anonymous access.

### How the Bridge Works

The `PrometheusARC3LiveEnv` wrapper translates between:

```
arc-agi toolkit            <->    Prometheus WP71
---------------------------------------------------
GameAction.ACTION1-7             ARC3Action (7 types)
FrameDataRaw (state)             ARC3Observation (grid)
GameState.WIN/GAME_OVER          episode.done
env.step(action, data={x,y})     agent.run_episode(env)
arc.get_scorecard()              ARC3BenchmarkReport
```

The Prometheus strange-loop architecture drives action selection; the toolkit
handles all networking, rendering, and official scorecard tracking.

### Action mapping

| Prometheus | arcengine | Meaning |
|------------|-----------|---------|
| `move_up` | `ACTION1` | Navigate upward |
| `move_down` | `ACTION2` | Navigate downward |
| `move_left` | `ACTION3` | Navigate leftward |
| `move_right` | `ACTION4` | Navigate rightward |
| `rotate` | `ACTION5` | Interact / select |
| `place` | `ACTION6` | Click at (x, y) — 0-63 grid |
| `undo` | `ACTION7` | Undo last action |


In [ ]:
# ── Install arc-agi toolkit ──────────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'arc-agi'],
    capture_output=True, text=True
)
print('arc-agi install:', 'OK' if result.returncode == 0 else result.stderr[:200])

# ── API key ───────────────────────────────────────────────────────────────────
# Paste your key from arcprize.org/platform, or leave blank for anonymous access
ARC_API_KEY = ''   # <-- replace with your key

if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f'API key configured ({ARC_API_KEY[:6]}...)')
else:
    print('No API key — anonymous access (3 public games: ls20, ft09, vc33)')
    print('Full access: register at arcprize.org/platform')


In [ ]:
# ── Prometheus <-> arc-agi bridge (v2: correct frame schema) ─────────────────
#
# Real ARC-AGI-3 API response schema (from arc3v1.yaml):
#   frame.frame          : list[list[list[int]]]  64×64 grid, values 0-15
#   frame.state          : GameState enum  (NOT_FINISHED / WIN / GAME_OVER / NOT_STARTED)
#   frame.levels_completed: int  cumulative levels completed this run
#   frame.win_levels     : int  levels required for WIN
#   frame.available_actions: list[int]  which action numbers are valid this turn
#
# What was broken in v1:
#   1. Grid was read from frame.state.get('grid') — wrong field, always None
#   2. Reward was frame.score delta — no such field, always 0
#   3. Done was comparing state dict to enum — never True, always hit step cap
#   4. Actions ignored available_actions — sent illegal actions every turn
# ─────────────────────────────────────────────────────────────────────────────
import random, time, math, json
from typing import Optional, Dict, Any, List

from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer,
    ARC3ExplorationPolicy, ARC3StrangeLoopAgent,
    _ACTION_TYPES,
)

try:
    import arc_agi
    from arcengine import GameAction, GameState
    TOOLKIT_AVAILABLE = True
    print('arc-agi toolkit ready.')
except ImportError as e:
    TOOLKIT_AVAILABLE = False
    print(f'arc-agi not available ({e}) — will use simulation fallback.')

# Bidirectional action mapping
_TO_GA = {
    'move_up':    'ACTION1',
    'move_down':  'ACTION2',
    'move_left':  'ACTION3',
    'move_right': 'ACTION4',
    'rotate':     'ACTION5',
    'place':      'ACTION6',
    'undo':       'ACTION7',
}
_FROM_GA_NUM = {1: 'move_up', 2: 'move_down', 3: 'move_left',
                4: 'move_right', 5: 'rotate', 6: 'place', 7: 'undo'}

# ARC-AGI-3 grid is 64×64 pixels; Prometheus coords are 0-4 (5×5 internal)
_SCALE = 63 // 4   # ≈ 15  →  0,15,30,45,60


def _frame_to_grid(frame) -> List[List[int]]:
    """
    Extract the 64×64 visual grid from a FrameDataRaw object.

    The API returns frame.frame as a list[list[list[int]]] —
    one or more visual frames, each 64×64, values 0-15 (4-bit colour).
    We take the first frame, downsample to 8×8 by block-averaging,
    and map values to 0-9 for compatibility with ARC3Observation.
    """
    raw = getattr(frame, 'frame', None)
    if raw is None or not raw:
        return [[0] * 8 for _ in range(8)]

    # raw may be shape [n_frames, 64, 64] or [64, 64] — normalise
    first = raw[0] if isinstance(raw[0][0], (list, tuple)) else raw
    h, w = len(first), len(first[0]) if first else 0
    if h == 0 or w == 0:
        return [[0] * 8 for _ in range(8)]

    # Downsample to 8×8 by taking every 8th row/col
    step_r = max(h // 8, 1)
    step_c = max(w // 8, 1)
    grid = []
    for r in range(0, min(h, 8 * step_r), step_r):
        row = []
        for c in range(0, min(w, 8 * step_c), step_c):
            val = first[r][c]
            # Handle nested lists (RGB channels)
            if isinstance(val, (list, tuple)):
                val = val[0]
            row.append(int(val) % 10)
        grid.append(row[:8])
    return grid[:8]


def _frame_to_obs(frame, prev_levels: int, step: int) -> tuple:
    """
    Convert a FrameDataRaw to (ARC3Observation, reward, done, levels_now,
    available_action_nums).

    Reward = levels_completed delta (completing a level = +1.0 reward).
    Done   = GameState.WIN or GAME_OVER.
    """
    if frame is None:
        return (
            ARC3Observation.from_grid_list([[0]*8]*8, score=0.0,
                                            done=False, step=step),
            0.0, False, prev_levels, list(range(1, 8))
        )

    grid = _frame_to_grid(frame)
    levels_now = int(getattr(frame, 'levels_completed', prev_levels) or prev_levels)
    reward = float(levels_now - prev_levels)   # +1 per level completed

    state = getattr(frame, 'state', None)
    done = False
    if TOOLKIT_AVAILABLE and state is not None:
        try:
            done = state in (GameState.WIN, GameState.GAME_OVER)
        except Exception:
            done = str(state).upper() in ('WIN', 'GAME_OVER')

    avail = getattr(frame, 'available_actions', None)
    if avail is None or not avail:
        avail = list(range(1, 8))
    avail = [int(a) for a in avail]

    obs = ARC3Observation.from_grid_list(
        grid,
        score=float(levels_now),
        done=done,
        step=step,
        metadata={'levels_completed': levels_now,
                  'available_actions': avail,
                  'state': str(state)},
    )
    return obs, reward, done, levels_now, avail


class PrometheusARC3LiveEnv:
    """
    Wraps arc-agi EnvironmentWrapper for ARC3StrangeLoopAgent.

    Key fixes vs v1:
    - Reads grid from frame.frame (64×64), downsampled to 8×8
    - Derives reward from levels_completed delta
    - Sets done from GameState enum comparison
    - Restricts actions to frame.available_actions
    - Rescales place (x,y) to 0-63 coordinate space
    """

    def __init__(self, env_wrapper, game_id: str, max_steps: int = 100):
        self._env = env_wrapper
        self.game_type = game_id
        self.max_steps = max_steps
        self._step = 0
        self._levels = 0
        self._done = False
        self._last_frame = None
        self._available = list(range(1, 8))   # default: all actions

    def reset(self) -> ARC3Observation:
        self._step = 0
        self._levels = 0
        self._done = False
        frame = self._env.reset()
        self._last_frame = frame
        obs, _, _, lvl, avail = _frame_to_obs(frame, 0, 0)
        self._levels = lvl
        self._available = avail
        return obs

    def step(self, action: ARC3Action):
        if self._done:
            obs, _, _, _, _ = _frame_to_obs(self._last_frame, self._levels, self._step)
            return obs, 0.0

        # Restrict to actions the environment currently permits
        ga_name = _TO_GA.get(action.action_type, 'ACTION1')
        ga_num = int(ga_name.replace('ACTION', ''))
        if ga_num not in self._available:
            # Fall back to first available action
            ga_num = self._available[0] if self._available else 1
            ga_name = f'ACTION{ga_num}'
            action = ARC3Action(
                action_type=_FROM_GA_NUM.get(ga_num, 'move_up'))

        action_data = None
        if ga_num == 6:   # ACTION6 requires (x, y) in 0-63 range
            action_data = {
                'x': max(0, min(63, action.x * _SCALE)),
                'y': max(0, min(63, action.y * _SCALE)),
            }

        try:
            ga = getattr(GameAction, ga_name)
            frame = self._env.step(ga, data=action_data)
            self._last_frame = frame
        except Exception as e:
            print(f'  [warn] step error ({ga_name}): {e}')
            frame = self._last_frame

        obs, reward, done, lvl, avail = _frame_to_obs(
            frame, self._levels, self._step)
        self._levels = lvl
        self._available = avail
        self._step += 1
        self._done = done or (self._step >= self.max_steps)
        return obs, reward

    @property
    def available_action_types(self) -> List[str]:
        return [_FROM_GA_NUM.get(n, 'move_up') for n in self._available]


def _run_synthetic_fallback(game_id, n_episodes, max_steps):
    from prometheus.wp71_arc_agi3 import _SyntheticARCGame
    synth_map = {'ls20': 'navigate', 'ft09': 'sort', 'vc33': 'mirror'}
    synth_type = synth_map.get(game_id, 'navigate')
    print(f'  [SIMULATION] proxy: {synth_type!r}')
    agent = ARC3StrangeLoopAgent(max_steps_per_episode=max_steps,
                                  mutation_rate=0.07, fitness_threshold=0.5)
    episodes = []
    for ep in range(n_episodes):
        env = _SyntheticARCGame(synth_type, grid_size=5,
                                max_steps=max_steps, seed=ep)
        episode = agent.run_episode(env)
        episodes.append(episode)
        print(f'  Ep {ep+1}/{n_episodes}: score={episode.total_score:.3f} '
              f'steps={episode.steps} solved={episode.solved}')
    sr = sum(1 for e in episodes if e.solved) / max(len(episodes), 1)
    return agent, episodes, sr, None, 'simulation'


def run_live_game(game_id='ls20', n_episodes=3, max_steps=100,
                  mutation_rate=0.07, fitness_threshold=0.5, verbose=True):
    """Run Prometheus ARC3StrangeLoopAgent on a live ARC-AGI-3 game."""
    if not TOOLKIT_AVAILABLE:
        agent, episodes, sr, api_sc, src = _run_synthetic_fallback(
            game_id, n_episodes, max_steps)
    else:
        print(f'Connecting to ARC-AGI-3 API [{game_id}]...')
        try:
            arc = arc_agi.Arcade()
            env_w = arc.make(game_id, render_mode=None)
            if env_w is None:
                raise RuntimeError('make() returned None')
        except Exception as e:
            print(f'  Connection failed: {e} — using simulation.')
            agent, episodes, sr, api_sc, src = _run_synthetic_fallback(
                game_id, n_episodes, max_steps)
        else:
            agent = ARC3StrangeLoopAgent(
                max_steps_per_episode=max_steps,
                mutation_rate=mutation_rate,
                fitness_threshold=fitness_threshold)
            live_env = PrometheusARC3LiveEnv(env_w, game_id, max_steps)
            episodes = []
            for ep in range(n_episodes):
                t0 = time.time()
                episode = agent.run_episode(live_env)
                episodes.append(episode)
                if verbose:
                    print(f'  Ep {ep+1}/{n_episodes}: '
                          f'levels={live_env._levels}  '
                          f'score={episode.total_score:.1f}  '
                          f'steps={episode.steps}  '
                          f'solved={episode.solved}  '
                          f'({time.time()-t0:.1f}s)')
            sr = sum(1 for e in episodes if e.solved) / max(len(episodes), 1)
            try:
                sc = arc.get_scorecard()
                api_sc = getattr(sc, 'score', None)
            except Exception:
                api_sc = None
            src = 'live_api'

    return {
        'game_id': game_id,
        'n_episodes': n_episodes,
        'solve_rate': sr,
        'mean_score': sum(e.total_score for e in episodes) / max(len(episodes), 1),
        'api_scorecard_score': api_sc,
        'world_model': agent.world_model.to_dict(),
        'goal_inferrer': agent.goal_inferrer.to_dict(),
        'policy': agent.policy.to_dict(),
        'entanglement_index': round(agent.entanglement_index, 4),
        'episodes': [e.to_dict() for e in episodes],
        'source': src,
    }

print('Bridge v2 ready.')
print(f'Toolkit available: {TOOLKIT_AVAILABLE}')

In [ ]:
# ── Run Prometheus on the 3 public ARC-AGI-3 games ───────────────────────────
#
# With the corrected bridge:
#   - levels_completed is the reward signal (completing a level = +1)
#   - done fires correctly on GameState.WIN / GAME_OVER
#   - only available_actions are sent to the environment
#
# Increase LIVE_MAX_STEPS if you want the agent to explore longer per episode.

PUBLIC_GAMES   = ['ls20', 'ft09', 'vc33']
LIVE_EPISODES  = 3
LIVE_MAX_STEPS = 100   # enough steps to complete at least one level

live_results = {}
t_start = time.time()

for game_id in PUBLIC_GAMES:
    print(f'\n{"="*55}')
    print(f'  Game: {game_id}')
    print('='*55)
    result = run_live_game(
        game_id=game_id,
        n_episodes=LIVE_EPISODES,
        max_steps=LIVE_MAX_STEPS,
        mutation_rate=0.07,
        fitness_threshold=0.5,
        verbose=True,
    )
    live_results[game_id] = result
    print(f'  --> solve rate: {result["solve_rate"]:.0%}  '
          f'mean score: {result["mean_score"]:.3f}  '
          f'entanglement: {result["entanglement_index"]:.3f}')

print(f'\nTotal time: {time.time()-t_start:.1f}s  |  '
      f'Source: {list(live_results.values())[0]["source"]}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math

def visualize_live_results(live_results: dict):
    games   = list(live_results.keys())
    colours = ['#3498db', '#e74c3c', '#2ecc71']
    n       = len(games)
    source  = list(live_results.values())[0].get('source', 'unknown')
    suffix  = '(Live API)' if source == 'live_api' else '(Simulation fallback)'

    fig = plt.figure(figsize=(20, 10))
    gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)
    ax_solve  = fig.add_subplot(gs[0, 0])
    ax_score  = fig.add_subplot(gs[0, 1])
    ax_radar  = fig.add_subplot(gs[0, 2], projection='polar')
    ax_policy = fig.add_subplot(gs[1, 0])
    ax_goal   = fig.add_subplot(gs[1, 1])
    ax_ent    = fig.add_subplot(gs[1, 2])

    # Solve rates
    sr = [live_results[g]['solve_rate'] * 100 for g in games]
    bars = ax_solve.bar(games, sr, color=colours[:n], edgecolor='black',
                        linewidth=1.2, alpha=0.85)
    ax_solve.set_ylabel('Solve rate (%)', fontsize=11, fontweight='bold')
    ax_solve.set_title(f'Solve Rate {suffix}', fontsize=12, fontweight='bold')
    ax_solve.set_ylim(0, 110); ax_solve.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, sr):
        ax_solve.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                      f'{v:.0f}%', ha='center', fontsize=12, fontweight='bold')

    # Mean scores
    ms = [live_results[g]['mean_score'] for g in games]
    ax_score.bar(games, ms, color=colours[:n], edgecolor='black',
                 linewidth=1.2, alpha=0.85)
    ax_score.set_ylabel('Mean episode score', fontsize=11, fontweight='bold')
    ax_score.set_title('Mean Score by Game', fontsize=12, fontweight='bold')
    ax_score.grid(True, alpha=0.3, axis='y')

    # Radar chart (first game)
    r0 = live_results[games[0]]
    dims = ['Exploration', 'Percept\n->Plan->Act', 'Memory',
            'Goal Acquis.', 'Alignment']
    policy_entropy = 1.0 - max(
        r0['policy']['strategy_probabilities'].values())
    vals = [policy_entropy,
            r0['world_model']['isomorphism_fidelity'],
            r0['solve_rate'],
            r0['goal_inferrer']['isomorphism_fidelity'],
            1.0]
    nd = len(dims)
    angles = [i * 2 * math.pi / nd for i in range(nd)] + [0]
    vp = vals + [vals[0]]
    ax_radar.plot(angles, vp, 'o-', linewidth=2.5, color='#e74c3c')
    ax_radar.fill(angles, vp, alpha=0.25, color='#e74c3c')
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(dims, fontsize=9, fontweight='bold')
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title(f'5 Dimensions ({games[0]})', fontsize=12,
                        fontweight='bold', pad=20)

    # Policy strategy probabilities (Good's mutation)
    x_pos = range(5)
    width = 0.25
    strategy_names = list(
        live_results[games[0]]['policy']['strategy_probabilities'].keys())
    for gi, (g, colour) in enumerate(zip(games, colours)):
        probs = list(
            live_results[g]['policy']['strategy_probabilities'].values())
        offset = (gi - 1) * width
        ax_policy.bar([x + offset for x in x_pos], probs,
                      width=width, alpha=0.8, label=g,
                      color=colour, edgecolor='black', linewidth=0.8)
    ax_policy.axhline(0.2, color='gray', linestyle='--', alpha=0.5,
                      label='Uniform')
    ax_policy.set_xticks(list(x_pos))
    ax_policy.set_xticklabels([s[:9] for s in strategy_names],
                               rotation=20, ha='right', fontsize=9)
    ax_policy.set_ylabel('Strategy probability', fontsize=11, fontweight='bold')
    ax_policy.set_title("Good's Mutation: Final Strategy Distribution",
                        fontsize=12, fontweight='bold')
    ax_policy.set_ylim(0, 1); ax_policy.legend(fontsize=8)
    ax_policy.grid(True, alpha=0.3, axis='y')

    # Goal confidence
    gc = [live_results[g]['goal_inferrer']['goal_confidence'] for g in games]
    gl = [live_results[g]['goal_inferrer']['most_likely_goal'] for g in games]
    bars2 = ax_goal.bar(games, gc, color=colours[:n], edgecolor='black',
                         linewidth=1.2, alpha=0.85)
    ax_goal.set_ylabel('Goal confidence', fontsize=11, fontweight='bold')
    ax_goal.set_title('Goal-Inferrer Confidence\n[Goal Acquisition]',
                      fontsize=12, fontweight='bold')
    ax_goal.set_ylim(0, 1.1); ax_goal.grid(True, alpha=0.3, axis='y')
    for bar, lbl in zip(bars2, gl):
        ax_goal.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.02, lbl,
                     ha='center', fontsize=8, rotation=12)

    # Entanglement index
    ev = [live_results[g]['entanglement_index'] for g in games]
    ax_ent.bar(games, ev, color=colours[:n], edgecolor='black',
               linewidth=1.2, alpha=0.85)
    ax_ent.set_ylabel('Entanglement index', fontsize=11, fontweight='bold')
    ax_ent.set_title('Strange-Loop Entanglement\n[World-Model <-> Policy]',
                     fontsize=12, fontweight='bold')
    ax_ent.set_ylim(0, 1.05); ax_ent.grid(True, alpha=0.3, axis='y')

    fig.suptitle(
        f'Prometheus WP71 — Live ARC-AGI-3 Results {suffix}',
        fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # Console summary
    print('\n' + '='*60)
    print('LIVE BENCHMARK SUMMARY')
    print('='*60)
    for g in games:
        r = live_results[g]
        print(f'  {g:6s}  solve={r["solve_rate"]:.0%}  '
              f'score={r["mean_score"]:.3f}  '
              f'ent={r["entanglement_index"]:.3f}  '
              f'goal={r["goal_inferrer"]["most_likely_goal"]}')
    api_sc = list(live_results.values())[0].get('api_scorecard_score')
    if api_sc is not None:
        print(f'\n  Official API scorecard: {api_sc}')
    print('='*60)


visualize_live_results(live_results)


In [ ]:
# ── (Optional) List all available ARC-AGI-3 games ────────────────────────────
# Requires an API key for the full catalogue; 3 games shown anonymously.

if TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        games = arc.get_environments()
        print(f'Available games ({len(games)} total):')
        for g in games:
            tags = getattr(g, 'tags', [])
            print(f'  {g.game_id:8s}  {g.title[:45]:45s}  tags={tags}')
    except Exception as e:
        print(f'Could not list games: {e}')
else:
    print('arc-agi toolkit not available.')
    print('Known public games:')
    print('  ls20  Agent Reasoning')
    print('  ft09  Elementary Logic')
    print('  vc33  Orchestration')
    print('Register at arcprize.org/platform for full access (API key required).')
